# Tests / Experiments
This notebook mostly contains some visualization / experiments on unrelated things in the `headset_localization` package.

In [ ]:
%load_ext autoreload
%autoreload 2
print(__debug__)

import logging
logging.basicConfig(level=logging.INFO)

import seaborn as sns
sns.set_theme(style="whitegrid", context="paper", font_scale=1)


import numpy as np
import random, torch, os, cv2

seed = 1

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
cv2.setRNGSeed(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ["PYTHONHASHSEED"] = str(seed)

import matplotlib.pyplot as plt

from headset_localization import *

In [ ]:
from shared.complete_robot_scan import CompleteRobotScan
robot_data_folder_location = "../example_datasets/small_aruco1/example_small_aruco1"
vrs_file_location = "../example_datasets/small_aruco1/small_aruco1_sitting_20fps.vrs"


robot_data = CompleteRobotScan.from_folder(robot_data_folder_location)
robot_env = Scanned3dEnvironment.from_gathered_robot_data(
        robot_data = robot_data,
        number_of_sampled_datapoints=10,
        sample_datapoints_based_on_aruco_corectness = False,
        only_sample_robot_datapoints_w_marker_estimates = True,
        markers_use_advanced_removal=True,
        est3d_xyz_image_gen_config = XYZImageGenerationConfig(iforest_contamination=0.05, use_depth_images_if_provided=True),
        est3d_xyz_icp_config=ICPAlignmentConfig(do_alginment=False)
)

labeled_headset_data = bind_headset_recording_to_scan(
        headset_data = HeadsetRecording.from_vrs_file(vrs_file_location),
        robot_data = robot_data
)

visualize_loaded_data = False

if visualize_loaded_data:
    visualize_robot_camera_environment_combo(robot_env=robot_env, headset_data=labeled_headset_data)


Simple usage example

In [ ]:
from headset_localization import HeadsetRecording, Scanned3dEnvironment, PnPLocalizer
from shared import CompleteRobotScan


headset_data = HeadsetRecording.from_vrs_file("../example_datasets/small_aruco1_sitting_20fps.vrs")

workspace_reconstruction = Scanned3dEnvironment.from_gathered_robot_data(
    robot_data = CompleteRobotScan.from_folder("../example_datasets/example_small_aruco1")
)

localizer = PnPLocalizer(
    cam2_intrinsic_mtx=headset_data.intrinsic_cam_mtx,
    cam1_bgr_images=workspace_reconstruction.robot_bgr_images,
    cam1_xyz_images=workspace_reconstruction.robot_xyz_images,
)

INDEX = 30
query_image = headset_data.bgr_image_s[INDEX]

base_t_cam = localizer.est_base_t_cam2(query_image)
print(base_t_cam)

In [ ]:
#pne_optimizer = PyposePNEOptimizer(PyposePnEOptimizerConfig())
#pne_optimizer = PnEDeltaPoseLBFGSOptimizer(time_tracker=tt_pne)

predictor = EllipsoidLocalizer(
        cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
        cam1_bgr_images=robot_env.robot_bgr_images,
        cam1_xyz_images=robot_env.robot_xyz_images,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
            rotation_augmentations=[Rotate180Deg],
            extract_and_match=ExtractAndLightGlue(),
            ransac_config=pose_estimation_ransaac_config_less_precise,
        ),
        pne_optimizer=PnEDeltaPoseAdamOptimizer(),
        ellipsoid_refinement_at_res=(1400, 1400),
        cam1_segmenter=SAM3Segmenter(Sam3Prompt()),
        cam2_segmenter=YOLOv26Segmenter("yoloe-26l-seg.pt"),
        matching_config=GaussianMatchingConfig(dummy_value=0.01),
        visualize_pne_optimisation=False,
        visualize_environment_generation=True,
        visualize_segmentation_masks=True
)

In [ ]:
ge = FastRayIntersectionError(points=robot_env.robot_xyz_images.reshape(-1,3), intrinsics=labeled_headset_data.intrinsic_cam_mtx, visualize=True)

grade = PredictionOnDataset(
    predictor=predictor,
    headset_data=labeled_headset_data
)

i, pred, label = grade.comparable_poses[0]

pixels = sample_pixel_neighborhood(center=(550,700))

fig, ax = plt.subplots(1,1)
ax.imshow(labeled_headset_data.bgr_image_s[i])
ax.scatter(pixels[:, 0], pixels[:,1], c='red', marker='o', s=2, label='Type A')

print(ge.calculate_gripping_differences_4_pixels(base_t_cam_s=np.array([pred, label]), pixels_batch=np.array([pixels, pixels])))

### Ellipsoid fitting

In [ ]:
import open3d as o3d

simple_fitter = SimpleEllipsoidFitter(visualize=False)
mvee_fitter = MVEEEllipsoidFitter(visualize=False, contamination=0.0)
ls_fitter = LeastShellDistanceEllipsoidFitter(visualize=False)

fitters = [simple_fitter, mvee_fitter, ls_fitter]

n_samples = 1000

points_x = np.random.rand(n_samples)-0.5
points_y = np.random.rand(n_samples)-0.5
points_z = -points_x**2 - points_y**2

pc = np.column_stack([points_x, points_y, points_z]) + 0.01 * np.random.rand(n_samples, 3)
print(f"pc: {pc.shape}")

for fitter in fitters:
    base_t_ellipsoid, primal_quadratic = fitter.fit_ellipsoid(points=pc)
    ls = create_ellipsoid_lineset(base_t_ellipsoid, primal_quadratic)

    pcd1 = o3d.geometry.PointCloud()
    pcd1.points = o3d.utility.Vector3dVector(pc)
    pcd1.paint_uniform_color([1,0,0])

    frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.05)
    frame.transform(base_t_ellipsoid)

    to_vis = [ls, pcd1, frame]
    o3d.visualization.draw_geometries(to_vis, f"Ellipsoid fit visualization", 1200, 1200)

In [ ]:
import open3d as o3d

simple_fitter = SimpleEllipsoidFitter(visualize=False)
mvee_fitter = MVEEEllipsoidFitter(visualize=False, contamination=0.0)
ls_fitter = LeastShellDistanceEllipsoidFitter(visualize=False, size_penalty=0.95)

fitters = [simple_fitter, mvee_fitter, ls_fitter]

n_samples = 1000

points_x = np.random.rand(n_samples)-0.5
points_y = np.random.rand(n_samples)-0.5
points_z = -points_x**2 - points_y**2

pc = np.column_stack([points_x, points_y, points_z]) + 0.01 * np.random.rand(n_samples, 3)
print(f"pc: {pc.shape}")

for fitter in fitters:
    base_t_ellipsoid, primal_quadratic = fitter.fit_ellipsoid(points=pc)
    ls = create_ellipsoid_lineset(base_t_ellipsoid, primal_quadratic)

    pcd1 = o3d.geometry.PointCloud()
    pcd1.points = o3d.utility.Vector3dVector(pc)
    pcd1.paint_uniform_color([1,0,0])

    frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.05)
    frame.transform(base_t_ellipsoid)

    to_vis = [ls, pcd1, frame]
    o3d.visualization.draw_geometries(to_vis, f"Ellipsoid fit visualization", 1200, 1200)

In [ ]:
from shared.complete_robot_scan import CompleteRobotScan
robot_data_folder_location = "../example_datasets/example_small_aruco1"
vrs_file_location = "../example_datasets/small_aruco1_sitting_20fps.vrs"


robot_data = CompleteRobotScan.from_folder(robot_data_folder_location)
n1 = 4
n = 6

robot_data_red = CompleteRobotScan(
    name=robot_data.name, 
    bgr_images=robot_data.bgr_images[n1:n],
    depth_images=robot_data.depth_images[n1:n],
    cam_intrinsic_mtx=robot_data.cam_intrinsic_mtx, 
    base_t_gripper_s=robot_data.base_t_gripper_s[n1:n],
    camera_t_marker_s=robot_data.camera_t_marker_s[n1:n], 
    marker_detector=robot_data.marker_detector, 
    gripper_t_cam=robot_data.gripper_t_cam
)


robot_env_red = Scanned3dEnvironment.from_gathered_robot_data(
        robot_data = robot_data_red,
        number_of_sampled_datapoints=n,
        sample_datapoints_based_on_aruco_corectness = False,
        only_sample_robot_datapoints_w_marker_estimates = False,
        markers_use_advanced_removal=True,
        est3d_xyz_image_gen_config = XYZImageGenerationConfig(iforest_contamination=0.05, use_depth_images_if_provided=True),
        est3d_xyz_icp_config=ICPAlignmentConfig(do_alginment=False)
)


img_rgb = cv2.cvtColor(robot_data_red.bgr_images[0], cv2.COLOR_BGR2RGB)
plt.imshow(img_rgb)
plt.axis('off')
plt.tight_layout(pad=0)
plt.show()

depth = robot_data_red.depth_images[0]
depth_normalized = (depth - depth.min()) / (depth.max() - depth.min())
plt.imshow(depth_normalized, cmap='gray')
plt.axis('off')
plt.tight_layout(pad=0)
plt.show()


img_rgb_no_marker = cv2.cvtColor(robot_env_red._robot_bgr_images[0], cv2.COLOR_BGR2RGB)
plt.imshow(img_rgb_no_marker)
plt.axis('off')
plt.tight_layout(pad=0)
plt.show()


robot_data_red.save("../img_scan", new_name="test")
robot_env_red.save("../img_env", new_name="test2")


world_points = robot_env_red.robot_xyz_images[0].reshape(-1,3)
world_colors = robot_env_red.robot_bgr_images[0].reshape(-1,3).astype(np.float32)[:, ::-1]/255
no_nan_mask = np.isfinite(world_points).all(axis=-1)

import open3d as o3d
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(world_points[no_nan_mask])
pcd.colors = o3d.utility.Vector3dVector(world_colors[no_nan_mask])
o3d.visualization.draw_geometries([pcd], f"Robot environment 1 image visualization")


labeled_headset_data = bind_headset_recording_to_scan(
        headset_data = HeadsetRecording.from_vrs_file(vrs_file_location),
        robot_data = robot_data
)

headset_img = cv2.cvtColor(labeled_headset_data.bgr_image_s[30], cv2.COLOR_BGR2RGB)
plt.imshow(headset_img)
plt.axis('off')
plt.tight_layout(pad=0)
plt.show()


In [ ]:
el = EllipsoidLocalizer(
    cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
    cam1_bgr_images=robot_env.robot_bgr_images,
    cam1_xyz_images=robot_env.robot_xyz_images, 
    cam1_segmenter=SAM3Segmenter(Sam3Prompt()),
    ellipsoid_fitter=LeastShellDistanceEllipsoidFitter(),
    visualize_environment_generation=True,
    visualize_segmentation_masks=True,
    visualize_pne_optimisation=True
)


visualize_primal_quadratics(
    base_t_ellipsoid_s=el.base_t_ellipsoid_s,
    primal_quadratic_s=el.primal_quadratic_s,
    avg_colors=plt.colormaps['viridis'](np.linspace(0, 1, el.base_t_ellipsoid_s.shape[0]))[:, :3]
)

el.est_base_t_cam2(cam2_bgr_image=labeled_headset_data.bgr_image_s[30])

In [ ]:
robot_data_folder_location_2 = "../example_datasets/small_aruco_2"
vrs_file_location_2 = "../example_datasets/small_aruco_2_2.vrs"


robot_data_2 = CompleteRobotScan.from_folder(robot_data_folder_location)
robot_env_2 = Scanned3dEnvironment.from_gathered_robot_data(
        robot_data = robot_data_2,
        number_of_sampled_datapoints=10,
        sample_datapoints_based_on_aruco_corectness = False,
        only_sample_robot_datapoints_w_marker_estimates = True,
        markers_use_advanced_removal=True,
        est3d_xyz_image_gen_config = XYZImageGenerationConfig(iforest_contamination=0.05, use_depth_images_if_provided=True),
        est3d_xyz_icp_config=ICPAlignmentConfig(do_alginment=False)
)

labeled_headset_data_2 = bind_headset_recording_to_scan(
        headset_data = HeadsetRecording.from_vrs_file(vrs_file_location),
        robot_data = robot_data_2
)

In [ ]:
lg = LineGenerator(line_cleanup_config=    
    MultiPassLineMergingConfig(passes=[
        LineMerging2dConfig(max_angle_diff = 2, max_midpoint_dist = 1/850, max_endpoint_dist = 0.02, min_line_length = 40/850, use_quick_merge = True),
        LineMerging2dConfig(max_angle_diff = 3, max_midpoint_dist = 2/850, max_endpoint_dist = 0.04, min_line_length = 40/850, use_quick_merge = True),
    ]),
    visualize_cleanup=True)
lg.get_lines(robot_env_2.robot_bgr_images[7])

### Comparison of different 3D reconstruction settings
This section does an ablation study, using the PnP LoMa predictor, of Depth and ICP Alignment and their influence on performance

In [ ]:
from shared.complete_robot_scan import CompleteRobotScan
from headset_localization import *

robot_data_folder_location = "../example_datasets/example_small_aruco1"
vrs_file_location = "../example_datasets/small_aruco1_sitting_20fps.vrs"


scans3d = []
names = []
predictors = []

robot_data = CompleteRobotScan.from_folder(robot_data_folder_location)
headset_recording = bind_headset_recording_to_scan(
        headset_data = HeadsetRecording.from_vrs_file(vrs_file_location),
        robot_data = robot_data
)

for use_depth, use_align in [(False, False), (False, True), (True, False), (True, True)]:
        print(f"creating: Depth: {use_depth}, Align: {use_align}")
        env3d = Scanned3dEnvironment.from_gathered_robot_data(
                robot_data = robot_data,
                number_of_sampled_datapoints=10,
                sample_datapoints_based_on_aruco_corectness = False,
                only_sample_robot_datapoints_w_marker_estimates = True,
                markers_use_advanced_removal=True,
                est3d_xyz_image_gen_config = XYZImageGenerationConfig(iforest_contamination=0.05, use_depth_images_if_provided=use_depth),
                est3d_xyz_icp_config=ICPAlignmentConfig(do_alginment=use_align)
        )
        scans3d.append(env3d)
        
        predictors.append(PnPLocalizer(
                        cam2_intrinsic_mtx=headset_recording.intrinsic_cam_mtx,
                        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
                                extract_and_match=ExtractAndMatchLoMa('LoMaB128')
                        ),
                        cam1_bgr_images=env3d.robot_bgr_images,
                        cam1_xyz_images=env3d.robot_xyz_images
                )
        )
        names.append(f"Depth: {use_depth}, Align: {use_align}")

import pandas as pd
import matplotlib.pyplot as plt

results = []

for name, predictor in zip(names, predictors):
    pod = PredictionOnDataset(predictor=predictor, headset_data=headset_recording)
    
    results.append({
        'Name': name,
        'Mean Error [mm]': np.mean(pod.translational_errors*1000),
        'Mean Error [deg]': np.rad2deg(np.mean(pod.rotational_errors)),
        'Median Error [mm]': np.median(pod.translational_errors)*1000,
        'Median Error [deg]': np.rad2deg(np.mean(pod.rotational_errors))
    })
df_results = pd.DataFrame(results)

In [ ]:
print(df_results)

fig, axes = plt.subplots(1,2, figsize = (12, 2.0))

NPredictors1DatasetGrader._plot_frontier_plot(
    df=df_results,
    xkey='Mean Error [mm]',
    ykey='Mean Error [deg]',
    title="Translational vs Rotational MAE",
    invert_y=False,
    name_key="Name",
    full_name_key="Name",
    ax=axes[0],
    adjust_texts=False,
    plot_names=False,
    plot_legend=True
)

NPredictors1DatasetGrader._plot_frontier_plot(
    df=df_results,
    xkey='Median Error [mm]',
    ykey='Median Error [deg]',
    title="Translational vs Rotational Median Errors",
    invert_y=False,
    name_key="Name",
    full_name_key="Name",
    ax=axes[1],
    adjust_texts=False,
    plot_names=False,
    plot_legend=False
)

plt.show()

## Vis Marker Removal

In [ ]:
import cv2

round1_scan = CompleteRobotScan.from_folder("../example_datasets/round1/round1_scan")
INDEX = 0

img_marker = round1_scan.bgr_images[INDEX]

round1_scan.marker_detector.set_new_masker("LamaMasker")
img_no_marker = round1_scan.marker_detector.remove_markers(round1_scan.bgr_images)[INDEX]

cv2.imwrite("image_with_marker.jpg", img_marker)
cv2.imwrite("image_without_marker.jpg", img_no_marker)

## Camera Alignment

In [ ]:
from headset_localization import *
from shared.complete_robot_scan import CompleteRobotScan


round1_scan = CompleteRobotScan.from_folder("../example_datasets/round1/round1_scan")

round1_env_na = Scanned3dEnvironment.from_gathered_robot_data(
        round1_scan, number_of_sampled_datapoints=100,
        est3d_xyz_image_gen_config = XYZImageGenerationConfig(iforest_contamination=0.1, use_depth_images_if_provided=True, camera_realginment_method = 'none'),
        est3d_xyz_icp_config = ICPAlignmentConfig()
)
round1_env_na.visualize_3d_data(visualize=True)

round1_env = Scanned3dEnvironment.from_gathered_robot_data(
        round1_scan, number_of_sampled_datapoints=100,
        est3d_xyz_image_gen_config = XYZImageGenerationConfig(iforest_contamination=0.1, use_depth_images_if_provided=True),
        est3d_xyz_icp_config = ICPAlignmentConfig()
)
round1_env.visualize_3d_data(visualize=True)


round1_rec2 = HeadsetRecording.from_folder("../example_datasets/round1/round1_rec2")

# ICP Alignment

In [ ]:
import open3d as o3d
import numpy as np
from headset_localization import *

n_samples = 1000


def gen_pc(n_samples = 1000, offsets:np.ndarray = np.array([0,0,0]), tilt_offsets:np.ndarray = np.array([0, 0])):
    points_x = np.random.rand(n_samples)-0.5 + offsets[0]
    points_y = np.random.rand(n_samples)-0.5 + offsets[1]
    points_z = 0.1 * (-points_x**2 - points_y**2) + offsets[2] + points_x * tilt_offsets[0] + points_y * tilt_offsets[1]
    pc = np.column_stack([points_x, points_y, points_z]) + 0.01 * np.random.rand(n_samples, 3)
    return pc

pc_s = [
    gen_pc(), 
    gen_pc(offsets=np.array([0.1, 0.0, 0.1]), tilt_offsets= np.array([0.1, 0])), 
    gen_pc(offsets=np.array([0.0, -0.1, -0.1]), tilt_offsets= np.array([-0.1, 0])), 
    gen_pc(offsets=np.array([0.0, 0.0, 0.0]), tilt_offsets= np.array([-0.3, 0])), 
]


pc_s_aligned = align_point_clouds_icp(point_clouds = pc_s, config = ICPAlignmentConfig(neighboar_dist_threshhold=0.1), visualize = False)

import matplotlib.cm as cm
import matplotlib.colors as colors

to_vis = []
norm = colors.Normalize(vmin=0, vmax=len(pc_s)-1)
for i, point_cloud in enumerate(pc_s):
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(point_cloud)
    pcd.paint_uniform_color(cm.jet(norm(i))[:3])
    to_vis.append(pcd)
o3d.visualization.draw_geometries(to_vis)

to_vis = []
norm = colors.Normalize(vmin=0, vmax=len(pc_s)-1)
for i, point_cloud in enumerate(pc_s_aligned):
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(point_cloud)
    pcd.paint_uniform_color(cm.jet(norm(i))[:3])
    to_vis.append(pcd)
o3d.visualization.draw_geometries(to_vis)

## Diverging LoMa, LightGlue

In [ ]:
from headset_localization import *
from shared.complete_robot_scan import CompleteRobotScan

round1_scan = CompleteRobotScan.from_folder("../example_datasets/round1/round1_scan")

round1_env = Scanned3dEnvironment.from_gathered_robot_data(
        round1_scan, number_of_sampled_datapoints=100,
        est3d_xyz_image_gen_config = XYZImageGenerationConfig(iforest_contamination=0.1, use_depth_images_if_provided=True),
        est3d_xyz_icp_config = ICPAlignmentConfig()
)

round1_rec1 = HeadsetRecording.from_folder("../example_datasets/round1/round1_rec1")

In [ ]:
import numpy as np

def predictors_for_dataset(intrinsic_mtx:np.ndarray)->list[GradableLocalizer]:
    points_light_glue = GradableLocalizer(
        creator= PnPLocalizer.get_creation_function(
            cam2_intrinsic_mtx=intrinsic_mtx,
            extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
                extract_and_match=ExtractAndLightGlue(),
                crop_augmentations=[0.4],
                scheduler=Scheduler
            )
        ),
        name="PnP-LG"
    )

    points_loma = GradableLocalizer(
        creator= PnPLocalizer.get_creation_function(
            cam2_intrinsic_mtx=intrinsic_mtx,
            extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
                extract_and_match=ExtractAndMatchLoMa('LoMaB128'),
                crop_augmentations=[0.2],
                scheduler=Scheduler
            )
        ),
        name="PnP-LoMa"
    )

    points_eloftr = GradableLocalizer(
        creator= PnPLocalizer.get_creation_function(
            cam2_intrinsic_mtx=intrinsic_mtx,
            extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
                extract_and_match=ExtractAndMatchEffLoFTR(),
                crop_augmentations=[0.2],
                scheduler = Scheduler
            )
        ),
        name="E-LOFTR"
    )

    return [points_light_glue, points_loma, points_eloftr]

def grader_for_dataset(env:Scanned3dEnvironment,headset_recording:HeadsetRecording)->NPredictors1DatasetGrader:
    return NPredictors1DatasetGrader(
        gradable_pose_predictors=predictors_for_dataset(headset_recording.intrinsic_cam_mtx),
        headset_data = headset_recording,
        robot_env = env,
        compute_ray_intersection_error=True, use_tqdm_for_frames=True, use_tqdm_for_predictors=False
    )

round1_rec1._n_frames = 80
round1_rec1._bgr_image_s = round1_rec1.bgr_image_s[40:120, :, :, :]
round1_rec1._robot_base_t_headset_s = round1_rec1._robot_base_t_headset_s[40:120]


round_rec2_grader = grader_for_dataset(env=round1_env, headset_recording=round1_rec1)


In [ ]:
import matplotlib.pyplot as plt
round_rec2_grader.print_summary()
_, ax = plt.subplots(1, 1, figsize = (8, 2.5), dpi = 500)
round_rec2_grader.plot_time_series_error(ax, TimeSeriesErrorType.ABS_TRANSLATIONAL, use_log_scale=True)

## Environment Generation

In [ ]:
from headset_localization import *
from shared.complete_robot_scan import CompleteRobotScan

brick1_scan = CompleteRobotScan.from_folder("../example_datasets/brick1/brick1_scan")

brick1_env = Scanned3dEnvironment.from_gathered_robot_data(
        brick1_scan, number_of_sampled_datapoints=100,
        est3d_xyz_image_gen_config = XYZImageGenerationConfig(iforest_contamination=0.1, use_depth_images_if_provided=True),
        est3d_xyz_icp_config = ICPAlignmentConfig()
)

brick1_rec1 = HeadsetRecording.from_folder("../example_datasets/brick1/brick1_rec1")

EllipsoidLocalizer(
    cam2_intrinsic_mtx=brick1_rec1.intrinsic_cam_mtx,
    cam1_bgr_images=brick1_env.robot_bgr_images,
    cam1_xyz_images=brick1_env.robot_xyz_images,
    #ellipsoid_fitter = MVEEEllipsoidFitter(contamination=0.1, visualize=False),
    ellipsoid_fitter=LeastShellDistanceEllipsoidFitter(contamination=0.02, size_penalty=0.05, size_p_norm=4),            
    extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
        extract_and_match=ExtractAndLightGlue(),
        crop_augmentations=[0.4],
    ),
    ellipsoid_matching_config=PointCloudMatchingConfig(min_cluster_size=2, max_color_dist=10),
    visualize_environment_generation = True,
    visualize_segmentation_masks = True,
    visualize_matching=False,
    visualize_pne_optimisation=False
)

# Matching Visualization

In [ ]:
from headset_localization import *
from shared.complete_robot_scan import CompleteRobotScan

robot_data_folder_location = "../example_datasets/small_aruco1/example_small_aruco1"
vrs_file_location = "../example_datasets/small_aruco1/small_aruco1_sitting_20fps.vrs"

robot_data = CompleteRobotScan.from_folder(robot_data_folder_location)
robot_env = Scanned3dEnvironment.from_gathered_robot_data(
        robot_data = robot_data,
        number_of_sampled_datapoints=10,
        sample_datapoints_based_on_aruco_corectness = False,
        only_sample_robot_datapoints_w_marker_estimates = True,
        markers_use_advanced_removal=True,
        est3d_xyz_image_gen_config = XYZImageGenerationConfig(iforest_contamination=0.05, use_depth_images_if_provided=True),
        est3d_xyz_icp_config=ICPAlignmentConfig(do_alginment=True)
)

labeled_headset_data = bind_headset_recording_to_scan(
        headset_data = HeadsetRecording.from_vrs_file(vrs_file_location),
        robot_data = robot_data
)

In [ ]:
ellipsoid_optimization_vis = EllipsoidLocalizer(
        cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
        cam1_bgr_images=robot_env.robot_bgr_images,
        cam1_xyz_images=robot_env.robot_xyz_images,
        matching_config=GaussianMatchingConfig(dummy_value=0.001),
        cam1_segmenter = SAM3Segmenter(Sam3Prompt(mask_threshold=0.2)),
        ellipsoid_fitter = MVEEEllipsoidFitter(contamination=0.2),
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
            extract_and_match=ExtractAndLightGlue(),
            crop_augmentations=[0.7],
        ),
        ellipsoid_matching_config=PointCloudMatchingConfig(min_cluster_size=2),
        visualize_matching=True,
        visualize_pne_optimisation=True
)

ellipsoid_predictor_grade = PredictionOnDataset(
    predictor = ellipsoid_optimization_vis,
    headset_data = labeled_headset_data,
)

In [ ]:
labeled_headset_data_mod = HeadsetRecording(
    name = "labeled_headset_data_mod",
    bgr_image_s=labeled_headset_data.bgr_image_s[10:12],
    intrinsic_cam_mtx=labeled_headset_data.intrinsic_cam_mtx,
    robot_base_t_headset_s=labeled_headset_data.robot_base_t_headset_s[10:12]
)

In [ ]:
ellipsoid_matching_vis = EllipsoidLocalizer(
        cam2_intrinsic_mtx=labeled_headset_data_mod.intrinsic_cam_mtx,
        cam1_bgr_images=robot_env.robot_bgr_images,
        cam1_xyz_images=robot_env.robot_xyz_images,
        matching_config=GaussianMatchingConfig(dummy_value=0.001),
        cam1_segmenter = SAM3Segmenter(Sam3Prompt(mask_threshold=0.2)),
        ellipsoid_fitter = MVEEEllipsoidFitter(contamination=0.2),
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
            extract_and_match=ExtractAndLightGlue(),
            crop_augmentations=[0.4],
        ),
        ellipsoid_matching_config=PointCloudMatchingConfig(min_cluster_size=2),
        visualize_matching=True,
        visualize_pne_optimisation=True
)

ellipsoid_predictor_grade = PredictionOnDataset(
    predictor = ellipsoid_matching_vis,
    headset_data = labeled_headset_data_mod,
)

In [ ]:
ellipsoid_video_predictor = EllipsoidLocalizer(
        cam2_intrinsic_mtx=labeled_headset_data_mod.intrinsic_cam_mtx,
        cam1_bgr_images=robot_env.robot_bgr_images,
        cam1_xyz_images=robot_env.robot_xyz_images,
        matching_config=GaussianMatchingConfig(dummy_value=0.001),
        cam1_segmenter = SAM3Segmenter(Sam3Prompt(mask_threshold=0.2)),
        ellipsoid_fitter = MVEEEllipsoidFitter(contamination=0.2),
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
            extract_and_match=ExtractAndLightGlue(),
            crop_augmentations=[0.4],
        ),
        ellipsoid_matching_config=PointCloudMatchingConfig(min_cluster_size=2),
        visualize_matching=True,
        visualize_pne_optimisation=True
)



ellipsoid_predictor_grade = PredictionOnDataset(
    predictor = ellipsoid_video_predictor,
    headset_data = labeled_headset_data_mod,
    vid_gen=VideoGenerator(
        fps=1, 
        style_config=FeatureStyleConfig(
            overlay_font_size = 20, unmatched_alpha=0.0, connection_line_alpha = 0.0, point_alpha=0.0, show_3d_points=False, show_ellipsoid_reprojection = False, line_widht=4
        ),
        scan_bgr_images=robot_env.robot_bgr_images,
        scan_xyz_images=robot_env.robot_xyz_images,
        use_second_3d_axis=True,
        figsize_3d=(labeled_headset_data_mod.bgr_image_s.shape[2], labeled_headset_data_mod.bgr_image_s.shape[1])
    ),
    video_save_location="ellipsoid_matching.mp4"
)

### PnP reference image selection

In [ ]:
mixed1_scan = CompleteRobotScan.from_folder("../example_datasets/mixed1/mixed1_scan")

mixed1_env = Scanned3dEnvironment.from_gathered_robot_data(
        mixed1_scan, number_of_sampled_datapoints=100,
        est3d_xyz_image_gen_config = XYZImageGenerationConfig(iforest_contamination=0.1, use_depth_images_if_provided=True),
        est3d_xyz_icp_config=ICPAlignmentConfig()
)

mixed1_rec1 = HeadsetRecording.from_folder("../example_datasets/mixed1/mixed1_rec1")
mixed1_rec2 = HeadsetRecording.from_folder("../example_datasets/mixed1/mixed1_rec2")

In [ ]:
mixed1_rec1_mod = HeadsetRecording(
    name = "labeled_headset_data_mod",
    bgr_image_s=mixed1_rec1.bgr_image_s[10:11],
    intrinsic_cam_mtx=mixed1_rec1.intrinsic_cam_mtx,
    robot_base_t_headset_s=mixed1_rec1.robot_base_t_headset_s[10:11]
)

In [ ]:
for i in range(8):
    pnp_video_predictor = PnPLocalizer(
            cam2_intrinsic_mtx=mixed1_rec1_mod.intrinsic_cam_mtx,
            cam1_bgr_images=mixed1_env.robot_bgr_images[i:i+1],
            cam1_xyz_images=mixed1_env.robot_xyz_images[i:i+1],
            extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(display_matching=False, crop_augmentations=[0.4]),
    )

    init_predictor_grade = PredictionOnDataset(
        predictor = pnp_video_predictor,
        headset_data = mixed1_rec1_mod,
        vid_gen=VideoGenerator(fps=1, 
            style_config=FeatureStyleConfig(
                connection_line_alpha=0.4, 
                point_size=6, 
                connection_line_thickness=2, 
                show_info_card=False, 
            ),
            use_second_3d_axis=False,
            figsize_3d=(mixed1_rec1_mod.bgr_image_s.shape[2], mixed1_rec1_mod.bgr_image_s.shape[1])
        ),
        video_save_location=f"pnp_ref_frame{i}.mp4"
    )
    init_predictor_grade.print_summary()

## Whole FR2/Desk

In [ ]:
dataset_location_fr2_desk = "../tum_datasets/rgbd_dataset_freiburg2_desk"

tum_robot_env, tum_headset_data = scanned_3d_environment_and_headset_recording_from_tum(
        folder=dataset_location_fr2_desk,
        rgb_camera_name="freiburg2",
        time_tolerance= 0.03,
        n_robot_images= 30,
        xyz_image_generation_config=XYZImageGenerationConfig(iforest_contamination = 0.5, use_depth_images_if_provided=False),
        xyz_image_alginment_config=ICPAlignmentConfig(do_alginment=True),
)

In [ ]:
visualize_robot_camera_environment_combo(robot_env=tum_robot_env, headset_data=tum_headset_data, vis_headset_camera_wireframes = False)

In [ ]:
pnp_loma_localizer_f = GradableLocalizer(
    creator=PnPLocalizer.get_creation_function(
        cam2_intrinsic_mtx=tum_headset_data.intrinsic_cam_mtx,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
            extract_and_match=ExtractAndMatchLoMa('LoMaB128'),
            rotation_augmentations=[Augmentation],
            display_matching=False,
            ransac_config = RansacPoseEstimationConfig(min_number_inlier_afterwards=70)
        )
    ),
    number_retries=3,
    name="PnP-LoMa"
)

grader = NPredictors1DatasetGrader(
    gradable_pose_predictors= [pnp_loma_localizer_f],
    compute_ray_intersection_error=True,
    headset_data = tum_headset_data,
    robot_env = tum_robot_env,
)

In [ ]:
grader.visualize_predictions_3d()